![Original climate-globe illustration introducing Heat, cold, and threshold indices; illustrative, not a data map](assets/atlas-splash.png)

# 05 · Heat, cold, and threshold indices

**Guiding question:** Can a monthly mean tell you how many days exceeded 35 °C?

**Data:** Synthetic teaching examples  
**Time:** 20–35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.  
**Author:** JupyterLite-C3S-Atlas contributors (AI assisted) · **Created/updated:** 2026-11-15  
**Inputs:** bundled reference files and/or seeded examples specified below. **Outputs:** tables, figures, and any explicitly named local exports; source data are not modified.  
**Runtime:** Python/Pyodide; NumPy, pandas, Matplotlib, Plotly; versions and tests are recorded in the validation report.  
**Change log:** initial educational chapter. **Contact:** [repository issues](https://github.com/jltobias/JupyterLite-C3S-Atlas/issues).

[Open in JupyterLite Lab](https://jltobias.github.io/JupyterLite-C3S-Atlas/lite/lab/index.html?path=05-heat-and-cold.ipynb) · [Official Atlas](https://atlas.climate.copernicus.eu/)


## Read the evidence

No: a mean loses the daily distribution. Count hot days from daily maxima and
       tropical nights from daily minima. A monthly maximum of daily maxima answers a
       different question from the monthly average of daily maxima.

       This invented daily year illustrates t, tn, tx, dtr, txx, tnn, tx35, tx40, tr,
       fd, cooling degree-days, and heating degree-days. Degree-days are temperature
       departures accumulated through time; they are not measured energy consumption.
       The base thresholds below follow the CDS catalogue's teaching definitions.

## Start the browser lab

Run this setup cell once. First load needs internet for the Python runtime and packages. The remaining lesson uses bundled data. If a 3D view is unavailable, use the 2D/table alternative. Download your edited notebook before clearing browser storage.

In [1]:
# Install only in the browser kernel; desktop dependencies come from requirements.txt.
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "plotly==6.5.0"])

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from atlas_helpers import (
    DATA, SYNTHETIC, COLORS, show, monthly_demo, anomalies, regular_grid,
    area_mean, coastlines, map_figure, globe_figure, warming_levels, load_monthly_csv,
)
print("Ready. Browser edits live in this browser; download work you want to keep.")


Ready. Browser edits live in this browser; download work you want to keep.


## Calculate from daily values

In [2]:
rng = np.random.default_rng(5)
dates = pd.date_range("2001-01-01", "2001-12-31", freq="D")
day = np.arange(len(dates))
mean_t = 15 + 13*np.cos(2*np.pi*(day-200)/365) + rng.normal(0, 2, len(day))
daily = pd.DataFrame({"date": dates, "t": mean_t, "tx": mean_t+7, "tn": mean_t-7})
summary = pd.Series({
    "tx35 days (>35 °C)": int((daily.tx > 35).sum()),
    "tx40 days (>40 °C)": int((daily.tx > 40).sum()),
    "tropical nights (>20 °C)": int((daily.tn > 20).sum()),
    "frost days (<0 °C)": int((daily.tn < 0).sum()),
    "cooling degree-days (base 22 °C)": np.maximum(daily.t-22, 0).sum(),
    "heating degree-days (base 15.5 °C)": np.maximum(15.5-daily.t, 0).sum(),
})
display(summary.round(1).to_frame("SYNTHETIC index value"))
assert summary.iloc[1] <= summary.iloc[0]

,SYNTHETIC index value
tx35 days (>35 °C),29.0
tx40 days (>40 °C),0.0
tropical nights (>20 °C),42.0
frost days (<0 °C),106.0
cooling degree-days (base 22 °C),490.1
heating degree-days (base 15.5 °C),1683.0


## See the information hidden by the mean

In [3]:
fig = go.Figure(go.Histogram(x=daily.tx, xbins=dict(size=2), name="Daily maxima"))
fig.add_vline(x=35, line_dash="dash", annotation_text="35 °C threshold")
fig.update_layout(title="Daily maximum distribution · SYNTHETIC", xaxis_title="Daily maximum temperature (°C)", yaxis_title="Days")
show(fig)
display(daily.assign(month=daily.date.dt.month).groupby("month").agg(
    tx_mean=("tx", "mean"), txx=("tx", "max"), tnn=("tn", "min")).round(2))

Figure({
    'data': [{'name': 'Daily maxima',
              'type': 'histogram',
              'x': {'bdata': ('zPEZgyDtH0CLg2W8Fn0bQLDtvas07S' ... '8k8yBAPMWPhi/8IEBUz+aX+gslQA=='),
                    'dtype': 'f8'},
              'xbins': {'size': 2}}],
    'layout': {'annotations': [{'showarrow': False,
                                'text': '35 °C threshold',
                                'x': 35,
                                'xanchor': 'left',
                                'xref': 'x',
                                'y': 1,
                                'yanchor': 'top',
                                'yref': 'y domain'}],
               'font': {'size': 14},
               'height': 470,
               'margin': {'b': 65, 'l': 65, 'r': 45, 't': 90},
               'shapes': [{'line': {'dash': 'dash'},
                           'type': 'line',
                           'x0': 35,
                           'x1': 35,
                           'xref': 'x',
                           'y0': 0,
                           'y1': 1,
                           'yref': 'y domain'}],
               'template': '...',
               'title': {'text': 'Daily maximum distribution · SYNTHETIC'},
               'xaxis': {'title': {'text': 'Daily maximum temperature (°C)'}},
               'yaxis': {'title': {'text': 'Days'}}}
})

,tx_mean,txx,tnn
month,,,
1,8.66,12.41,-8.45
2,10.08,16.90,-7.86
3,14.42,18.06,-2.58
4,20.85,25.98,1.06
5,28.14,33.67,8.29
6,32.27,37.09,13.02
7,35.30,39.95,16.79
8,33.69,39.78,14.62
9,29.05,32.88,9.52


## Try it and explain

Add 1 °C to every daily maximum and recompute tx35. Why is the count response not necessarily linear?

**Interpretation check:** Only days near the threshold cross it. The response depends on the distribution, not just its mean. Bias adjustment therefore matters strongly for threshold counts.

## Transfer to the official tools

Compare t, txx, and tx35 for the same region. Check whether bias adjustment is available and whether the units change.

Record the selections, source version, units, periods, and uncertainty legend. These independent lessons do not duplicate every production workflow in the official tools.

## Sources and credit

- [C3S Atlas dataset and DOI](https://doi.org/10.24381/cds.h35hb680)
- [CSIC / Predictia Atlas user guide](https://confluence.ecmwf.int/spaces/CKB/pages/394237545/Copernicus+Interactive+Climate+Atlas+User+Guide)
- [Official C3S Atlas User Tools and contributors](https://ecmwf-projects.github.io/c3s-atlas/)

Original teaching text and code; original AI-generated splash illustration. Real reference files retain their source licences. Synthetic examples are not Copernicus measurements. See [credits](credits.md) and [data guide](data-guide.md).

**Continue:** [Rainfall, extremes, and the water cycle](06-rain-and-water.ipynb)